# 04 — Explore

**Exploration only.** Tries the question several ways to find the honest framing.
Nothing here is final — `06-viz-social` is built only after owner review.

**Angles in this notebook:**
1. Self (I/me/my) vs collective (we/us/our) — over time + per-president, three lenses.
2. Straight **word counts** — avg speech length by president (within a lens).
3. **Tenure-weighted** speech volume (control for 4 vs 8-yr terms, partial terms).
4. **Common vs distinctive** words — raw frequency vs TF-IDF, and a **per-president
   word-cloud picker** (`docs/presidents.html`, like the countries picker).

**⭐ Apples-to-oranges guard:** the corpus is curated + coverage is uneven, so
per-president comparisons are made *within one speech type* (whole corpus = context
only; SOTU series; Inaugural), and the SOTU written(≤1912)/spoken break is respected.

## ⚠️ Rendering note
Py3.14 venv → matplotlib RecursionErrors, so exploration uses the **shared Pillow
factory** (what social export also uses). Charts render inline below.

In [ ]:
import sys, os
from pathlib import Path

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT.parent.parent / 'shared'))

# Word clouds need the `wordcloud` package. Fail with a clear message if the
# active kernel's environment is missing it, rather than a deep import error.
try:
    import wordcloud  # noqa: F401
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "'wordcloud' is not installed in this kernel's environment. "
        "Install it with:  pip install wordcloud"
    )

import duckdb, pandas as pd
from chart_factory import render_chart
from colors import c

DB = 'data/project.duckdb'
print('using', DB)

## Build exploration aggregation tables

Derive per-lens `chart_*` tables from `speeches_clean` so this notebook is
self-contained. Writable connection for the builds, then reconnect read-only to render.

In [ ]:
w = duckdb.connect(DB)
w.execute('DROP TABLE IF EXISTS chart_sotu_by_decade')
w.execute("""CREATE TABLE chart_sotu_by_decade AS
  SELECT CAST(FLOOR(year/10.0)*10 AS INTEGER) AS decade, COUNT(*) n,
         ROUND(AVG(self_per_1k),2) self_1k, ROUND(AVG(collective_per_1k),2) coll_1k,
         ROUND(AVG(self_share),3) self_share
  FROM speeches_features WHERE is_sotu_series GROUP BY 1 ORDER BY decade""")
# Ranked tables: single_ranked_bars renders rows top->bottom in DataFrame order and
# does NOT sort, so we ORDER BY value DESC here (biggest bar on top).
w.execute('DROP TABLE IF EXISTS chart_sotu_self_share')
w.execute("""CREATE TABLE chart_sotu_self_share AS
  SELECT president, COUNT(*) n_sotu, ROUND(AVG(self_share),3) self_share,
         printf('%.0f%%  (%d)', AVG(self_share)*100, COUNT(*)) AS label
  FROM speeches_features WHERE is_sotu_series GROUP BY 1 HAVING COUNT(*)>=2
  ORDER BY self_share DESC""")
w.execute('DROP TABLE IF EXISTS chart_inaug_self_share')
w.execute("""CREATE TABLE chart_inaug_self_share AS
  SELECT president, ROUND(AVG(self_share),3) self_share,
         printf('%.0f%%', AVG(self_share)*100) AS label
  FROM speeches_features WHERE speech_type='Inaugural Address' GROUP BY 1
  ORDER BY self_share DESC""")
w.execute('DROP TABLE IF EXISTS chart_sotu_length')
w.execute("""CREATE TABLE chart_sotu_length AS
  SELECT president, COUNT(*) n, CAST(ROUND(AVG(word_count),0) AS INTEGER) avg_words,
         printf('%,d', CAST(ROUND(AVG(word_count),0) AS INTEGER)) AS label
  FROM speeches_features WHERE is_sotu_series GROUP BY 1 HAVING COUNT(*)>=2
  ORDER BY avg_words DESC""")
w.execute('DROP TABLE IF EXISTS chart_speeches_per_year')
w.execute("""CREATE TABLE chart_speeches_per_year AS
  SELECT president, speeches_per_year,
         printf('%.1f/yr', speeches_per_year) AS label
  FROM president_terms WHERE reliable_rate
  ORDER BY speeches_per_year DESC""")
for t in ['chart_sotu_by_decade','chart_sotu_self_share','chart_inaug_self_share',
          'chart_sotu_length','chart_speeches_per_year']:
    print(t, w.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0], 'rows')

# Diverging (top10/bottom10) variants — CHART-TYPE EXPLORATION to compare vs the
# ranked bars above. lean = self_share - 0.5 (right=leans "I", left=leans "we");
# label shows the actual self-share % so every bar reads as the same metric.
for _src, _dst in [('chart_sotu_self_share','chart_sotu_lean_tb'),
                   ('chart_inaug_self_share','chart_inaug_lean_tb')]:
    w.execute(f'DROP TABLE IF EXISTS {_dst}')
    w.execute(f"""CREATE TABLE {_dst} AS
      WITH r AS (SELECT president, self_share,
             ROW_NUMBER() OVER (ORDER BY self_share DESC) rt,
             ROW_NUMBER() OVER (ORDER BY self_share ASC) rb FROM {_src})
      SELECT president, ROUND(self_share-0.5,3) AS lean,
             printf('%.0f%% “I”', self_share*100) AS label
      FROM r WHERE rt<=10 OR rb<=10 ORDER BY self_share DESC""")

w.close()
con = duckdb.connect(DB, read_only=True)
print('reconnected read-only')

## ⭐ Over-time (SOTU) — the strongest framing: the rise of “we”

Self vs collective per 1,000 words, by decade. **Key finding:** the gap is driven by
*collective* language climbing steeply in the broadcast era — self stays comparatively
flat. Honest headline = “presidents invoke ‘we’ far more than they used to,” NOT “more
self-focused.” Reads across the written(≤1912)/spoken break.

In [ ]:
render_chart({
    'type': 'line', 'db': con, 'table': 'chart_sotu_by_decade', 'x_col': 'decade',
    'series': [{'col': 'coll_1k', 'label': 'collective (we/us/our)', 'color': c('teal')},
               {'col': 'self_1k', 'label': 'self (I/me/my)', 'color': c('spice')}],
    'x_axis_label': 'Decade', 'y_axis_label': 'Pronouns per 1,000 words',
    'legend': True, 'label_last': False, 'y_min': 0, 'x_tick_step': 20,
    'title': 'State of the Union — self vs collective pronoun rate by decade',
    'subtitle': 'Avg per 1,000 words across each decade’s SOTUs (Annual Message + State of the Union). Pre-1913 were written messages read by a clerk.',
    'source': 'Miller Center (UVA) speech archive',
    'filename': 'explore_sotu_self_vs_coll_by_decade',
})

## Per-president self_share — SOTU and Inaugural lenses

self ÷ (self + collective); 50% = balanced. Apples-to-apples within each lens. The
early/written-era presidents rank highest — which is the time-trend restated.

In [ ]:
render_chart({
    'type': 'single_ranked_bars', 'db': con, 'table': 'chart_sotu_self_share',
    'category_col': 'president', 'value_col': 'self_share', 'label_col': 'label',
    'bar_color': c('teal'), 'height': 1500,
    'title': 'State of the Union — share of first-person pronouns that are “I” not “we”',
    'subtitle': 'self ÷ (self+collective), avg per president (≥2 SOTUs). Label: share (n).',
    'source': 'Miller Center (UVA) speech archive',
    'filename': 'explore_sotu_self_share_ranked',
})

In [ ]:
render_chart({
    'type': 'single_ranked_bars', 'db': con, 'table': 'chart_inaug_self_share',
    'category_col': 'president', 'value_col': 'self_share', 'label_col': 'label',
    'bar_color': c('caramel'), 'height': 1500,
    'title': 'Inaugural Address — share of first-person pronouns that are “I” not “we”',
    'subtitle': 'self ÷ (self+collective) per inaugural (avg if two). 50% = balanced.',
    'source': 'Miller Center (UVA) speech archive',
    'filename': 'explore_inaug_self_share_ranked',
})

## ⭐ CHART-TYPE OPTION: same data as diverging bars (from 50/50)

Same self_share metric as the two ranked charts above, but drawn as **top-10 /
bottom-10 diverging** from a 50/50 centerline: right of center leans “I” (teal),
left leans “we” (gold); every bar labeled with its actual “% I”. Compare this to
the ranked form above and tell me which reads better per lens. (SOTU comes out
lopsided — only ~3 presidents clear 50%; inaugural diverges more evenly.)


In [ ]:
render_chart({
    'type': 'diverging_bars', 'db': con, 'table': 'chart_sotu_lean_tb',
    'category_col': 'president', 'value_col': 'lean', 'label_col': 'label',
    'pos_color': c('teal'), 'neg_color': c('gold'), 'zero_label': '50/50', 'sort': True,
    'title': 'SOTU — “I” vs “we” (diverging option)',
    'subtitle': 'Every bar = share of first-person pronouns that are “I”. Right leans “I”, left leans “we”. Top 10 & bottom 10 (≥2 SOTUs).',
    'source': 'Miller Center (UVA) speech archive',
    'filename': 'explore_sotu_lean_diverging',
})

In [ ]:
render_chart({
    'type': 'diverging_bars', 'db': con, 'table': 'chart_inaug_lean_tb',
    'category_col': 'president', 'value_col': 'lean', 'label_col': 'label',
    'pos_color': c('teal'), 'neg_color': c('gold'), 'zero_label': '50/50', 'sort': True,
    'title': 'Inaugural — “I” vs “we” (diverging option)',
    'subtitle': 'Every bar = share of first-person pronouns that are “I”. Right leans “I”, left leans “we”. Top 10 & bottom 10.',
    'source': 'Miller Center (UVA) speech archive',
    'filename': 'explore_inaug_lean_diverging',
})

## Word counts — avg SOTU length by president (owner ask)

Straight length, within the SOTU lens. This is really the written/spoken divide again:
the long ones are the pre-1913 *written* messages (Taft ~22k words), the short ones are
*spoken* addresses (Washington ~2k). A length chart must caption that break.

In [ ]:
render_chart({
    'type': 'single_ranked_bars', 'db': con, 'table': 'chart_sotu_length',
    'category_col': 'president', 'value_col': 'avg_words', 'label_col': 'label',
    'bar_color': c('navy'), 'height': 1500,
    'title': 'State of the Union — average length by president (words)',
    'subtitle': 'Longest are pre-1913 WRITTEN messages; shortest are spoken addresses. ≥2 SOTUs.',
    'source': 'Miller Center (UVA) speech archive',
    'filename': 'explore_sotu_length',
})

## Tenure-weighted volume (owner ask: control for term length)

Speeches per year in office = n_speeches ÷ (days_in_office / 365.25), so 8-year and
4-year presidencies compare fairly. Short-tenure presidents (<1 yr: W. Harrison,
Garfield) are EXCLUDED — a rate off ~1 speech in a few weeks is a tiny-denominator
artifact (`reliable_rate` flag).

**Important caveat:** this is a rate over the CURATED corpus — how many of a president's
speeches the Miller Center *included* per year — NOT their true speaking output. It
measures coverage density, so it leans modern. Useful as an internal check on why
raw per-president counts mislead; probably NOT a shippable social chart.

In [ ]:
render_chart({
    'type': 'single_ranked_bars', 'db': con, 'table': 'chart_speeches_per_year',
    'category_col': 'president', 'value_col': 'speeches_per_year', 'label_col': 'label',
    'bar_color': c('gold'), 'height': 1500,
    'title': 'Speeches in the corpus per year in office',
    'subtitle': 'CORPUS COVERAGE rate (curated inclusion ÷ tenure), not total output. Tenures ≥1yr only.',
    'source': 'Miller Center (UVA) speech archive + curated term dates',
    'filename': 'explore_speeches_per_year',
})

## Most-used vs distinctive words (owner ask) + the per-president picker

Two different questions:
- **Most-used words** (raw frequency) — this president's own most-frequent words.
  Every president's top words overlap a lot (people, nation, states) — shown for Lincoln.
- **Distinctive words** (TF-IDF) — what a president says far more than *other* presidents;
  their defining vocabulary. More revealing (Reagan: soviet, nuclear, mondale).

Clouds show only words the president SPOKE — transcription cues ([Applause]/[Laughter])
are excluded and charted separately below. Below: Lincoln most-used vs distinctive.


In [ ]:
lincoln_common = con.execute("""SELECT word, count FROM word_freq_by_president
  WHERE president='Abraham Lincoln' ORDER BY count DESC LIMIT 60""").df()
render_chart({
    'type': 'word_cloud', 'table': lincoln_common, 'word_col': 'word', 'weight_col': 'count',
    'title': 'Abraham Lincoln — most-used words (raw frequency)',
    'subtitle': 'His most-frequent spoken words (common function words removed).',
    'source': 'Miller Center (UVA) speech archive', 'filename': 'explore_wc_lincoln_mostused',
})

In [ ]:
lincoln_distinct = con.execute("""SELECT word, tfidf FROM distinctive_words_by_president
  WHERE president='Abraham Lincoln' ORDER BY tfidf DESC LIMIT 60""").df()
render_chart({
    'type': 'word_cloud', 'table': lincoln_distinct, 'word_col': 'word', 'weight_col': 'tfidf',
    'title': 'Abraham Lincoln — DISTINCTIVE words (TF-IDF vs other presidents)',
    'subtitle': 'Words he used far more than others — his defining vocabulary.',
    'source': 'Miller Center (UVA) speech archive', 'filename': 'explore_wc_lincoln_distinct',
})

## Applause & laughter in the room — a TV-era artifact

When the Miller Center transcribes a delivered speech it inserts markers like
`[Applause]` and `[Laughter]` **where the audience actually reacted during delivery**
— these are transcriber annotations of the live reaction, not text written into the
speech. They aren't spoken words, so they're kept OUT of the clouds. Their own story:
they appear almost only in the broadcast era. Presidents with ≥20 reaction markers,
applause (incl. “applauding”) vs laughter (incl. “laughs”). Cheers/boos are a
handful of stray cases; “inaudible” is a transcription gap, not a reaction — both
excluded here.


In [ ]:
# build the chart table: sum applause-family + laughter-family (complete totals);
# exclude 'inaudible' (a transcription gap, not a reaction) and 'cheering' (mostly the
# adjective in older prose, not a [Cheering] marker). Presidents with >=20 reactions.
con.close()
w = duckdb.connect(DB)
w.execute('DROP TABLE IF EXISTS chart_stage_directions')
w.execute("""CREATE TABLE chart_stage_directions AS
  SELECT president,
         (applause + applauding) AS applause,
         (laughter + laughs)     AS laughter,
         (applause + applauding + laughter + laughs + cheers + booing) AS reaction
  FROM stage_directions_by_president
  WHERE (applause + applauding + laughter + laughs + cheers + booing) >= 20
  ORDER BY reaction DESC""")
w.close()
con = duckdb.connect(DB, read_only=True)
render_chart({
    'type': 'stacked_bars', 'db': con, 'table': 'chart_stage_directions',
    'category_col': 'president',
    'segments': [
        {'value_col': 'applause', 'label': 'Applause', 'color': c('teal')},
        {'value_col': 'laughter', 'label': 'Laughter', 'color': c('gold')},
    ],
    'title': 'Applause and laughter in presidential speeches',
    'subtitle': 'Transcriber markers of live audience reaction per president (>=20). Almost entirely a TV-era phenomenon.',
    'source': 'Miller Center (UVA) speech archive',
    'filename': 'explore_stage_directions',
})

## ⭐ Each president's #1 most-used word, over time (owner ask)

The most-used clouds are dominated by “states” for early presidents — accurate, but it
hides the arc. Here's the arc itself: each president's single most-frequent word, in order,
colored by word-family. **“states” is #1 for every president 1789→1909 (Washington→Taft),
then never again** — it gives way to “world/peace” (mid-century superpower era) and
“people” (FDR onward). Same ~1913 Wilson pivot as the rise-of-we and written→spoken breaks.


In [ ]:
# build the two tables (writable conn, then reconnect read-only)
con.close()
w = duckdb.connect(DB)
TEAL, GOLD, RUST, GRAY = c('teal'), c('gold'), c('spice'), c('gray')
w.execute('DROP TABLE IF EXISTS chart_top_word_timeline')
w.execute(f"""CREATE TABLE chart_top_word_timeline AS
  WITH ranked AS (
    SELECT wf.president, wf.word, wf.count,
      ROW_NUMBER() OVER (PARTITION BY wf.president ORDER BY wf.count DESC) rn,
      MIN(s.year) OVER (PARTITION BY wf.president) fy
    FROM word_freq_by_president wf JOIN speeches_clean s ON s.president=wf.president)
  SELECT fy || '  ·  ' || president AS row_label, word AS top_word, count, fy,
    CASE WHEN word IN ('states','union') THEN '{{TEAL}}'
         WHEN word IN ('world','peace') THEN '{{GOLD}}'
         WHEN word IN ('people','country') THEN '{{RUST}}'
         ELSE '{{GRAY}}' END AS color,
    word AS label
  FROM ranked WHERE rn=1 ORDER BY fy""".format(TEAL=TEAL,GOLD=GOLD,RUST=RUST,GRAY=GRAY))
w.execute('DROP TABLE IF EXISTS chart_top_word_family')
w.execute("""CREATE TABLE chart_top_word_family AS
  SELECT CASE WHEN top_word IN ('states','union') THEN 'states / union'
              WHEN top_word IN ('world','peace') THEN 'world / peace'
              WHEN top_word IN ('people','country') THEN 'people / country'
              ELSE 'other' END AS family, COUNT(*) n,
         printf('%d presidents', COUNT(*)) AS label
  FROM chart_top_word_timeline GROUP BY 1 ORDER BY n DESC""")
w.close()
con = duckdb.connect(DB, read_only=True)

# chronological (sort=None), bar length = count of the top word, colored by family
render_chart({
    'type': 'single_ranked_bars', 'db': con, 'table': 'chart_top_word_timeline',
    'category_col': 'row_label', 'value_col': 'count', 'label_col': 'label',
    'color_col': 'color', 'sort': None, 'height': 1500,
    'legend': [{'label':'states/union','color':TEAL},{'label':'world/peace','color':GOLD},
               {'label':'people/country','color':RUST},{'label':'other','color':GRAY}],
    'title': 'Every president’s most-used word, 1789→present',
    'subtitle': 'Top word per president (chronological). “States” dominates 1789–1909, then “world” then “people.”',
    'source': 'Miller Center (UVA) speech archive',
    'filename': 'explore_top_word_timeline',
})

### Most-used word by era (summary)

The same thing counted up: how many presidents had each word-family as their #1 word.


In [ ]:
render_chart({
    'type': 'single_ranked_bars', 'db': con, 'table': 'chart_top_word_family',
    'category_col': 'family', 'value_col': 'n', 'label_col': 'label',
    'bar_color': c('teal'),
    'title': 'What presidents talk about most — by word family',
    'subtitle': 'Count of presidents whose single most-used word falls in each family.',
    'source': 'Miller Center (UVA) speech archive',
    'filename': 'explore_top_word_family',
})

con.close()  # release the read lock before the script opens its own connection
import subprocess
# Runs on the SAME interpreter as this kernel (sys.executable); needs wordcloud
# in that environment (pip install wordcloud). Regenerates docs/presidents/*.png
# + docs/presidents.html.
r = subprocess.run([sys.executable, 'scripts/build_president_pages.py'],
                   capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr[-800:])
con = duckdb.connect(DB, read_only=True)
print('Open docs/presidents.html in a browser to use the picker.')
print('45 clouds in docs/presidents/. Sample (Reagan):')
from IPython.display import Image as _Img
_Img(filename='docs/presidents/ronald-reagan.png', width=800)

## Notes for owner review (before `06-viz-social`)

- **Lead chart candidate:** the over-time SOTU line, reframed around the **rise of “we”**
  (self is roughly flat; collective drives the story). Descriptive vs finding-led title?
- **self_share rankings** (SOTU, inaugural): defensible within a lens, but top = 19th-c.
  presidents = the time trend restated. Lead, secondary, or drop?
- **Word length** chart is really the written/spoken divide — interesting but needs the
  caveat front-and-center; maybe better as a written-vs-spoken framing than a ranking.
- **Speeches-per-year** = corpus-coverage artifact, not output — recommend NOT shipping;
  it's here to show why raw counts mislead.
- **Word clouds:** distinctive (TF-IDF) is far more interesting than common. The
  per-president picker (`docs/presidents.html`) is a strong interactive artifact.
  Caveat: some distinctive tokens are opponent/moderator surnames from debate
  transcripts (Reagan→mondale, walters) — could filter debates or add a stopword pass.
- **Caveats every final chart carries:** curated corpus; written(≤1912)/spoken break;
  ghostwriting (speech as delivered).

---
## Cleanup
Close the DuckDB connection so the lock is released for other tools.

In [ ]:
con.close()
print('connection closed')